In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Helps reduce PyTorch memory fragmentation.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
import json
import re
import time
from pathlib import Path
from typing import Optional

import torch
import transformers
import vllm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm.auto import tqdm

print("Python executable:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
print("CUDA visible device count:", torch.cuda.device_count())
print("CUDA version:", torch.version.cuda)
print("Torch version:", torch.__version__)
print("transformers:", transformers.__version__)
print("vLLM:", vllm.__version__)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"cuda:{i} ->", torch.cuda.get_device_name(i))
else:
    raise RuntimeError("CUDA is not available. You are not in a GPU pod/session.")

Python executable: /home/folin/CSE 151B/151B_SP26_Competition/.venv/bin/python
CUDA available: True
CUDA visible device count: 1
CUDA version: 12.1
Torch version: 2.5.1+cu121
transformers: 5.7.0
vLLM: 0.7.3
cuda:0 -> NVIDIA A30 MIG 2g.12gb


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

DATA_PATH = "data/public.jsonl"

RUN_NAME = "prompt_v2_greedy_smoke_50"
# # K=1 baseline (sampling, no voting)
# RUN_NAME = "prompt_v2_sc_k1_50";  n=1

# # K=3 self-consistency
# RUN_NAME = "prompt_v2_sc_k3_50";  n=3

# # K=5 self-consistency
# RUN_NAME = "prompt_v2_sc_k5_50";  n=5
OUTPUT_PATH = f"results/{RUN_NAME}.jsonl"

# Conservative first. After it works, raise this to 8192.
MAX_TOKENS = 8192

# Start with 10. After model loads + scores correctly, change to 50.
EVAL_LIMIT = 50

print("MODEL_ID:", MODEL_ID)
print("DATA_PATH:", DATA_PATH)
print("RUN_NAME:", RUN_NAME)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("MAX_TOKENS:", MAX_TOKENS)
print("EVAL_LIMIT:", EVAL_LIMIT)

MODEL_ID: Qwen/Qwen3-4B-Thinking-2507
DATA_PATH: data/public.jsonl
RUN_NAME: prompt_v2_greedy_smoke_50
OUTPUT_PATH: results/prompt_v2_greedy_smoke_50.jsonl
MAX_TOKENS: 8192
EVAL_LIMIT: 50


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [3]:
data_path = Path(DATA_PATH)
assert data_path.exists(), f"Cannot find {DATA_PATH}. Run this notebook from the competition repo root."

data = [json.loads(line) for line in open(data_path, encoding="utf-8")]

if EVAL_LIMIT is None:
    eval_data = data
else:
    eval_data = data[:EVAL_LIMIT]

n_mcq_all  = sum(bool(d.get("options")) for d in data)
n_free_all = sum(not d.get("options") for d in data)

n_mcq_eval  = sum(bool(d.get("options")) for d in eval_data)
n_free_eval = sum(not d.get("options") for d in eval_data)

print(f"Loaded {len(data)} total questions  ({n_mcq_all} MCQ, {n_free_all} free-form)")
print(f"Evaluating {len(eval_data)} questions ({n_mcq_eval} MCQ, {n_free_eval} free-form)")

# Preview one MCQ and one free-form item from eval_data
mcq_sample  = next(d for d in eval_data if d.get("options"))
free_sample = next(d for d in eval_data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2)[:1500])
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2)[:1500])


Loaded 1126 total questions  (375 MCQ, 751 free-form)
Evaluating 50 questions (13 MCQ, 37 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [4]:
# Goal: force a final boxed answer while avoiding endless reasoning loops.

SYSTEM_PROMPT_FREEFORM = """
You are a careful math solver.

Solve the problem step by step, but keep the reasoning concise.
Do not stop before giving the final answer.

Formatting rules:
1. The final line must be exactly: Therefore, the answer is \\boxed{...}
2. Put only the final answer content inside \\boxed{}.
3. If the problem has multiple [ANS] blanks, put the answers in order, separated by commas.
4. Do not use words like "approximately" unless the problem asks for an approximation.
""".strip()

SYSTEM_PROMPT_MCQ = """
You are a careful math solver.

Solve the multiple-choice problem step by step, but keep the reasoning concise.
Compare your result to the answer choices.

Formatting rules:
1. The final line must be exactly: Therefore, the answer is \\boxed{X}
2. X must be one capital letter such as A, B, C, D, or E.
3. Do not put the full option text inside \\boxed{}.
""".strip()

# SYSTEM_PROMPT_FREEFORM = (
#     "You are an expert mathematician. Solve this problem using a detailed Chain of Thought. "
#     "Break the problem into logical sub-tasks. At the end of each step, validate your reasoning "
#     "to ensure no calculation or conceptual errors have occurred. "
#     "If you find an inconsistency, backtrack and correct it immediately. "
#     "After thorough verification, put your final answer inside \boxed{}. "
#     "If there are multiple sub-answers, separate them by commas inside the \boxed{}, e.g. \boxed{3, 7}."
# )
# SYSTEM_PROMPT_MCQ = (
#     "You are an expert mathematician. Use a Chain of Thought approach to solve this "
#     "multiple-choice problem. First, solve the problem independently without looking at the "
#     "choices. Then, compare your derived result against the provided options. "
#     "Validate each step of your deduction. If your result does not match any option, "
#     "re-examine your assumptions. Output ONLY the capital letter of the single best "
#     "answer inside \boxed{}, e.g. \boxed{C}."
# )


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for one competition item."""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(
            f"{label}. {str(option).strip()}"
            for label, option in zip(labels, options)
        )

        user_prompt = f"""
Problem:
{question}

Answer choices:
{opts_text}

Solve the problem and end with the required boxed letter.
""".strip()

        return SYSTEM_PROMPT_MCQ, user_prompt

    user_prompt = f"""
Problem:
{question}

Solve the problem and end with the required boxed answer.
""".strip()

    return SYSTEM_PROMPT_FREEFORM, user_prompt


# Verify with samples
# for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
#     sys_p, usr_p = build_prompt(item["question"], item.get("options"))
#     print("=" * 80)
#     print(label)
#     print("SYSTEM PROMPT:")
#     print(sys_p)
#     print("\nUSER PROMPT PREVIEW:")
#     print(usr_p[:1000])


## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [5]:
# ── Load tokenizer + patch Qwen tokenizer compatibility ──────────────────────
from transformers.models.qwen2.tokenization_qwen2 import Qwen2Tokenizer

if not hasattr(Qwen2Tokenizer, "all_special_tokens_extended"):
    print("Patching Qwen2Tokenizer.all_special_tokens_extended ...")

    @property
    def all_special_tokens_extended(self):
        return list(self.all_special_tokens)

    Qwen2Tokenizer.all_special_tokens_extended = all_special_tokens_extended
else:
    print("Qwen2Tokenizer already has all_special_tokens_extended.")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="left",
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer class:", tokenizer.__class__)
print("Has all_special_tokens_extended:", hasattr(tokenizer, "all_special_tokens_extended"))

# ── Load vLLM model ───────────────────────────────────────────────────────────
vllm_model = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    trust_remote_code=True,
    gpu_memory_utilization=0.90,
    max_model_len=12288,
    max_num_seqs=2,
    max_num_batched_tokens=12288,
    enable_prefix_caching=True,
)
print("Model loaded.")

Patching Qwen2Tokenizer.all_special_tokens_extended ...
Tokenizer class: <class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>
Has all_special_tokens_extended: True
INFO 05-03 18:29:46 __init__.py:207] Automatically detected platform cuda.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


INFO 05-03 18:29:56 config.py:549] This model supports multiple tasks: {'score', 'reward', 'embed', 'generate', 'classify'}. Defaulting to 'generate'.
INFO 05-03 18:29:56 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.3) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=10240, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=Qwen/Qwen3-4B-Thinking-2507, n

[W503 18:29:58.068371063 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


INFO 05-03 18:29:59 weight_utils.py:254] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-03 18:30:00 model_runner.py:1115] Loading model weights took 7.4925 GB
INFO 05-03 18:30:03 worker.py:267] Memory profiling takes 2.91 seconds
INFO 05-03 18:30:03 worker.py:267] the current vLLM instance can use total_gpu_memory (11.69GiB) x gpu_memory_utilization (0.90) = 10.52GiB
INFO 05-03 18:30:03 worker.py:267] model weights take 7.49GiB; non_torch_memory takes 0.04GiB; PyTorch activation peak memory takes 0.77GiB; the rest of the memory reserved for KV Cache is 2.22GiB.
INFO 05-03 18:30:04 executor_base.py:111] # cuda blocks: 1011, # CPU blocks: 1820
INFO 05-03 18:30:04 executor_base.py:116] Maximum concurrency for 10240 tokens per request: 1.58x
INFO 05-03 18:30:09 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_util

Capturing CUDA graph shapes: 100%|██████████| 2/2 [00:01<00:00,  1.66it/s]

INFO 05-03 18:30:10 model_runner.py:1562] Graph capturing finished in 1 secs, took 0.31 GiB
INFO 05-03 18:30:10 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 9.96 seconds
Model loaded.


In [6]:
# Greedy decoding for prompt experiments.
sampling_params_sc = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.7,
    top_p=0.95,
    n=1,                          # K=1, K=3, K=5
    repetition_penalty=1.0,
)

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [7]:
# ── Generate K samples per question ───────────────────
def format_chat_prompt(item: dict) -> str:
    system, user = build_prompt(item["question"], item.get("options"))
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

prompts = [format_chat_prompt(item) for item in eval_data]

print(f"Built {len(prompts)} prompts. K={sampling_params_sc.n}")
print(f"Generating {len(prompts) * sampling_params_sc.n} total samples...")

outputs = vllm_model.generate(prompts, sampling_params=sampling_params_sc)

# Keep ALL K samples per question, plus token-level diagnostics.
per_question_raw = []
for out in outputs:
    samples = []
    for o in out.outputs:
        samples.append({
            "text": o.text.strip(),
            "n_tokens": len(o.token_ids),
            "finish_reason": o.finish_reason,   # "stop" or "length" (=truncated)
        })
    per_question_raw.append(samples)

assert len(per_question_raw) == len(eval_data)

K = len(per_question_raw[0])
print(f"\nGeneration complete. K={K}")
print(f"Sample 0, response 0 preview:\n{per_question_raw[0][0]['text'][:400]}")
print(f"\nFinish reasons (first question): {[s['finish_reason'] for s in per_question_raw[0]]}")

Built 50 prompts. K=1
Generating 50 total samples...


Processed prompts:  24%|██▍       | 12/50 [12:06<50:20, 79.49s/it, est. speed input: 4.29 toks/s, output: 65.42 toks/s]   

WARNING 05-03 18:46:02 scheduler.py:1754] Sequence group 13 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=1


Processed prompts: 100%|██████████| 50/50 [1:30:15<00:00, 108.31s/it, est. speed input: 2.49 toks/s, output: 63.68 toks/s]


Generation complete. K=1
Sample 0, response 0 preview:
Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me recall what the first few even whole numbers are. The positive even whole numbers start at 2, right? So the first one is 2, the second is 4, the third is 6, and so on. 

I remember that the sum of the first n even numbers can be found using a formula. Let me think. The nth even number is 2n, ri

Finish reasons (first question): ['stop']


In [8]:
def extract_boxed(text: str):
    """
    Extract the last \\boxed{...} content.
    This handles nested braces like \\boxed{\\frac{1}{2}}, unlike a simple regex.
    """
    marker = r"\boxed{"
    start = text.rfind(marker)
    if start == -1:
        return None

    i = start + len(marker)
    depth = 1
    chars = []

    while i < len(text):
        ch = text[i]

        if ch == "{":
            depth += 1
            chars.append(ch)
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return "".join(chars).strip()
            chars.append(ch)
        else:
            chars.append(ch)

        i += 1

    return None


# for i in range(min(5, len(responses))):
#     print("=" * 80)
#     print("id:", eval_data[i].get("id"))
#     print("boxed:", extract_boxed(responses[i]))
#     print("response length:", len(responses[i]))
#     print("tail:")
#     print(responses[i][-1000:])


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [16]:
# ── Helper: extract MCQ letter ─────────────────────────
import re
import sys
from tqdm import tqdm
import string

def get_allowed_letters(item):
    """
    Infer allowed MCQ letters from item['options'].
    Supports options as list or dict.
    """
    options = item.get("options", None)

    if isinstance(options, dict):
        keys = [str(k).strip().upper() for k in options.keys()]
        letter_keys = [k for k in keys if len(k) == 1 and k in string.ascii_uppercase]
        if letter_keys:
            return "".join(letter_keys)
        return string.ascii_uppercase[:len(options)]

    if isinstance(options, list):
        return string.ascii_uppercase[:len(options)]

    # Fallback: many math MCQ datasets use A-J
    return string.ascii_uppercase[:10]


def extract_letter(text, allowed=None):
    """
    Extract final MCQ letter from model text.
    Returns one uppercase letter or None.
    """
    if text is None:
        return None

    if allowed is None:
        allowed = string.ascii_uppercase[:10]

    allowed = "".join([c for c in allowed.upper() if c in string.ascii_uppercase])
    char_class = re.escape(allowed)

    s = str(text).strip()

    # Clean common LaTeX wrappers
    s = re.sub(r"\\text\{([^{}]*)\}", r"\1", s)
    s = s.replace("$", "")

    # Prefer boxed answer
    boxed_matches = re.findall(
        rf"\\boxed\s*\{{\s*\(?\s*([{char_class}])\s*\)?\s*\}}",
        s,
        flags=re.IGNORECASE,
    )
    if boxed_matches:
        return boxed_matches[-1].upper()

    upper_s = s.upper()

    # Prefer explicit final-answer language near the end
    patterns = [
        rf"(?:THEREFORE|THUS|SO|FINAL ANSWER|ANSWER|THE ANSWER IS)\s*(?:IS|:)?\s*\(?\s*([{char_class}])\s*\)?",
        rf"\bOPTION\s+([{char_class}])\b",
        rf"\bCHOICE\s+([{char_class}])\b",
    ]

    candidates = []
    for pat in patterns:
        for m in re.finditer(pat, upper_s):
            candidates.append((m.start(), m.group(1).upper()))

    if candidates:
        # take the last explicit answer-like mention
        return sorted(candidates, key=lambda x: x[0])[-1][1]

    # Only use this if the whole text is basically just a letter
    m = re.fullmatch(rf"\s*\(?\s*([{char_class}])\s*\)?\.?\s*", upper_s)
    if m:
        return m.group(1).upper()

    return None


# ── Vote + score + record diagnostics ─────────────────
def majority_vote(boxed_answers):
    """Return (voted_answer, status). Status: 'majority' | 'tie_first' | 'all_none'."""
    valid = [b for b in boxed_answers if b is not None]
    if not valid:
        return None, "all_none"

    counts = {}
    for b in valid:
        counts[b] = counts.get(b, 0) + 1

    max_count = max(counts.values())
    winners = {b for b, c in counts.items() if c == max_count}

    if len(winners) == 1:
        return next(iter(winners)), "majority"

    # Tie: deterministic — first occurrence wins
    for b in valid:
        if b in winners:
            return b, "tie_first"


sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []

for item, samples in tqdm(zip(eval_data, per_question_raw), total=len(eval_data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold = item.get("answer", None)

    sample_texts = [s["text"] for s in samples]
    sample_boxed = [extract_boxed(t) for t in sample_texts]

    if K == 1:
        voted, vote_status = sample_boxed[0], "k1"
    else:
        voted, vote_status = majority_vote(sample_boxed)

    # Pick representative trace: first sample whose boxed answer matches the vote
    if voted is not None:
        rep_idx = next((i for i, b in enumerate(sample_boxed) if b == voted), 0)
    else:
        rep_idx = 0

    rep_text = sample_texts[rep_idx]

    # Score against voted answer when possible
    if gold is None:
        correct = None

    elif is_mcq:
        allowed_letters = get_allowed_letters(item)
    
        # First extract from voted boxed answer
        pred_letter = extract_letter(voted, allowed=allowed_letters) if voted is not None else None
    
        # Fall back to representative response
        if pred_letter is None:
            pred_letter = extract_letter(rep_text, allowed=allowed_letters)
    
        correct = (pred_letter == str(gold).strip().upper())

    else:
        gold_list = gold if isinstance(gold, list) else [gold]

        # For free-form, use voted boxed answer if available.
        # If there is no boxed answer, use the representative full response.
        pred_for_judge = voted if voted is not None else rep_text

        try:
            correct = judger.auto_judge(
                pred=pred_for_judge,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    # Diagnostics across K samples
    has_boxed_per = [b is not None for b in sample_boxed]
    truncated_per = [s["finish_reason"] == "length" for s in samples]
    n_tokens_per = [s["n_tokens"] for s in samples]

    results.append({
        "id": item.get("id"),
        "is_mcq": is_mcq,
        "gold": gold,
        "K": K,
        "samples_boxed": sample_boxed,
        "voted": voted,
        "vote_status": vote_status,
        "rep_response": rep_text,
        "pred_letter": pred_letter if is_mcq else None,
        "correct": correct,
        "any_has_boxed": any(has_boxed_per),
        "all_have_boxed": all(has_boxed_per),
        "any_truncated": any(truncated_per),
        "all_truncated": all(truncated_per),
        "tokens_per_sample": n_tokens_per,
        "max_tokens_used": max(n_tokens_per),
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 50/50 [00:17<00:00,  2.94it/s]

Scoring complete. 50 results.


## 8. Summary

Print accuracy broken down by question type.

In [17]:
scored_results = [r for r in results if r["correct"] is not None]
mcq_res  = [r for r in scored_results if r["is_mcq"]]
free_res = [r for r in scored_results if not r["is_mcq"]]

def acc(subset):
    return sum(bool(r["correct"]) for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 60)
print("EVALUATION RESULTS")
print("RUN_NAME:", RUN_NAME)
print("=" * 60)
print(f"  MCQ        : {sum(bool(r['correct']) for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(bool(r['correct']) for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(bool(r['correct']) for r in scored_results):4d} / {len(scored_results):4d}  ({acc(scored_results):.2f}%)")
print("=" * 60)


EVALUATION RESULTS
RUN_NAME: prompt_v2_greedy_smoke_50
  MCQ        :    6 /   13  (46.15%)
  Free-form  :    1 /   37  (2.70%)
  Overall    :    7 /   50  (14.00%)


In [18]:
# ── Formatting diagnostics (Table 11 in milestone report) ────────────────────
def format_diagnostics(results):
    n = len(results)
    K = results[0]["K"] if results else 0
    has_any   = sum(1 for r in results if r["any_has_boxed"])
    has_all   = sum(1 for r in results if r["all_have_boxed"])
    miss_all  = sum(1 for r in results if not r["any_has_boxed"])
    trunc_any = sum(1 for r in results if r["any_truncated"])
    trunc_all = sum(1 for r in results if r["all_truncated"])
    ties      = sum(1 for r in results if r.get("vote_status") == "tie_first")
    none_vote = sum(1 for r in results if r.get("vote_status") == "all_none")
    avg_tok   = sum(sum(r["tokens_per_sample"]) / len(r["tokens_per_sample"]) for r in results) / n
    pct = lambda x: f"{x}/{n} ({x/n*100:.1f}%)"
    return {
        "RUN_NAME": RUN_NAME,
        "n": n, "K": K,
        "Has Boxed (any sample)":  pct(has_any),
        "Has Boxed (all samples)": pct(has_all),
        "Missing Boxed (all)":     pct(miss_all),
        "Truncated (any sample)":  pct(trunc_any),
        "Truncated (all samples)": pct(trunc_all),
        "Vote ties (K>1 only)":    pct(ties)     if K > 1 else "n/a",
        "All-None votes":          pct(none_vote) if K > 1 else "n/a",
        "Avg tokens/sample":       round(avg_tok, 1),
    }

diag = format_diagnostics(results)
print("=" * 70)
print("FORMATTING DIAGNOSTICS")
print("=" * 70)
for k, v in diag.items():
    print(f"  {k:30s} : {v}")
print("=" * 70)

FORMATTING DIAGNOSTICS
  RUN_NAME                       : prompt_v2_greedy_smoke_50
  n                              : 50
  K                              : 1
  Has Boxed (any sample)         : 16/50 (32.0%)
  Has Boxed (all samples)        : 16/50 (32.0%)
  Missing Boxed (all)            : 34/50 (68.0%)
  Truncated (any sample)         : 35/50 (70.0%)
  Truncated (all samples)        : 35/50 (70.0%)
  Vote ties (K>1 only)           : n/a
  All-None votes                 : n/a
  Avg tokens/sample              : 6896.9


In [19]:
print("len(data):", len(data))
print("len(eval_data):", len(eval_data))
print("len(prompts):", len(prompts))
print("len(per_question_raw):", len(per_question_raw))
print("len(results):", len(results))

# Show wrong examples to diagnose prompt failures.
wrong = [r for r in results if r["correct"] is False]

for r in wrong[:5]:
    print("=" * 100)
    print(
        "id:", r["id"],
        "is_mcq:", r["is_mcq"],
        "gold:", r["gold"],
        "voted:", r["voted"],
        "pred_letter:", r.get("pred_letter"),
        "vote_status:", r["vote_status"],
        "any_truncated:", r["any_truncated"],
        "max_tokens_used:", r["max_tokens_used"],
    )
    print("representative response tail:")
    print(r["rep_response"][-1200:])

len(data): 1126
len(eval_data): 50
len(prompts): 50
len(per_question_raw): 50
len(results): 50
id: 1 is_mcq: True gold: F voted: C pred_letter: C vote_status: k1 any_truncated: False max_tokens_used: 5432
representative response tail:
ndard result from calculus. The integral of $ \frac{1}{s^2 + a^2} $ over the entire real line is:

$$
\int_{-\infty}^{+\infty} \frac{1}{s^2 + a^2} \, ds = \frac{\pi}{a}
$$

So the full expression becomes:

$$
a^{3/2} \cdot \frac{\pi}{a} = \pi \cdot a^{1/2}
$$

However, **none of the answer choices include $ \pi $**. This suggests that either the problem has a typo or there's an implicit assumption in the context (e.g., approximate values).

---

Let’s examine the answer choices:

- A. $ 0 $
- B. $ \frac{1}{a} $
- C. $ \frac{3}{a} $
- D. $ \frac{1}{2a^2} $
- E. $ \frac{1}{2a} $
- F. $ \frac{2}{a} $
- G. $ 2a $
- H. $ \frac{3}{2a} $
- I. $ \frac{3}{2a^2} $
- J. $ \frac{1}{a^2} $

The correct value is $ \pi \cdot a^{1/2} $, which for $ a = 1 $ gives approxim

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [20]:
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Saved {len(results)} records to {out_path}")

Saved 50 records to results/prompt_v2_greedy_smoke_50.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!